# Project B — SAEs for concept control in Stable Diffusion (unattended)

The whole pipeline runs from **one cell** and doesn't need you watching it.

### On Kaggle (recommended)
1. Sidebar → **Accelerator: GPU T4 x2**, **Internet: On**.
2. Top right → **Save Version** → **Save & Run All (Commit)** → Save.
3. Close the tab. This runs as a background job, so there's no idle disconnect. The limit is 12 hours per run; run 2 takes roughly 4–6 (twice the images of run 1, plus 3 seeds for steering).
4. Come back to **your notebook → Versions → the run → Output / Logs**. `project_b/outputs/REPORT.md` has every number.

### If a stage fails
The notebook doesn't stop with an error. It finishes cleanly so the version **saves everything completed so far**. To continue from the failed stage instead of starting over:
open the notebook → **Add Input → Your Work → Notebooks →** this notebook → pick the failed version. Then **Save & Run All** again.
The configuration cell finds the attached output and resumes. Finished stages are skipped, and the activations are only copied if training hadn't finished.

It uses both T4s: captures run side by side, and the five SAEs are split across the two GPUs.

### Adding the erasure experiment to a finished run (about 1–1.5 hours)
Do **not** start from scratch. Open the notebook → **Add Input → Your Work → Notebooks →** this notebook →
pick the version that produced run 2's results. Then **Save & Run All**. Every finished stage is skipped (no
capture, no training); only `erase_multi` runs, one SAE per GPU, and `REPORT.md` is rebuilt with the new section.
If several earlier versions are attached, the one captured at the current `SITE` is used.

### On Colab
Runtime → T4 GPU → Run all. It mounts Google Drive and keeps SAEs and results there, so if the
runtime disconnects **after training**, Run all again picks up where it stopped. Activations stay on the VM's local disk. A disconnect *during* capture or training
redoes those stages, because free Colab runtimes don't survive a disconnect. You only get one GPU, so it's roughly twice as slow as Kaggle.
Free Colab also has no background execution — keep the tab open.

In [ ]:
!pip install -q 'diffusers>=0.27' 'transformers>=4.38' accelerate safetensors
import torch, subprocess
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout)


In [ ]:
import base64, io, tarfile, os, pathlib, sys
REPO_B64 = (
    "..."
)
BASE = '/kaggle/working' if os.path.exists('/kaggle/working') else ('/content' if os.path.exists('/content') else os.getcwd())
repo = pathlib.Path(BASE) / 'diffusion-sae'
repo.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(REPO_B64)), mode='r:gz') as tf:
    try:
        tf.extractall(repo, filter='data')
    except TypeError:
        tf.extractall(repo)
os.chdir(repo); sys.path.insert(0, str(repo))
print('repo at', repo)


### Go / no-go: test suite (CPU, ~3 min)
All three must pass, or don't spend GPU time.

In [ ]:
import subprocess, sys
for t in ('tests/test_core.py', 'tests/test_stages.py', 'tests/test_pipeline.py'):
    r = subprocess.run([sys.executable, t], capture_output=True, text=True)
    print(t, '->', r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-500:])
    assert r.returncode == 0, r.stdout[-3000:] + r.stderr[-3000:]


### Configuration
The defaults are sized for Kaggle T4 x2. On Colab, `N_IMAGES` drops to fit one GPU and the VM's disk.

In [ ]:
import os, sys, pathlib
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules

if ON_KAGGLE:
    ROOT = '/kaggle/working/project_b'          # saved as the version's output
    ACTS = ROOT + '/data'
    N_IMAGES = 1000
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/project_b'   # survives disconnects
    ACTS = '/content/sae_acts'                  # fast local disk; random reads from Drive are slow
    N_IMAGES = 600
else:
    ROOT = str(pathlib.Path.cwd() / 'project_b_run'); ACTS = ROOT + '/data'; N_IMAGES = 500

# Run 2 captures the transformer block's residual stream. Run 1 used the attn2
# sub-layer, whose output turned out to be low-rank (82-92% dead features).
# Set SITE = ATTN2 only to reproduce run 1.
RESIDUAL = 'up_blocks.1.attentions.1.transformer_blocks.0'
ATTN2 = RESIDUAL + '.attn2'
SITE = RESIDUAL

# Delete the ~8 GB of captured activations once everything has finished. Keeps
# the saved Kaggle output small. Set True if you want to retrain later
# without recapturing.
KEEP_ACTS = False

# Resume source: this session's own previous progress is picked up automatically
# (same ROOT). A previous *version's* output can be attached via Add Input.
import glob
RESUME_FROM = None
if ON_KAGGLE:
    import json as _json
    def _site(st):
        f = os.path.join(st, 'capture_info.json')
        return _json.load(open(f))['site'] if os.path.exists(f) else ATTN2
    found = sorted(glob.glob('/kaggle/input/**/project_b/state', recursive=True))
    match = [st for st in found if _site(st) == SITE]
    # Prefer the attached run with the most finished stages at this SITE.
    match.sort(key=lambda st: len(glob.glob(os.path.join(st, '*.done'))), reverse=True)
    if match:
        RESUME_FROM = os.path.dirname(match[0])
    elif found:
        print('attached outputs were captured at a different site -- not resuming from them')
print(f'ROOT={ROOT}\nACTS={ACTS}\nSITE={SITE}\nN_IMAGES={N_IMAGES}\nRESUME_FROM={RESUME_FROM}')


## Run everything

One command, eleven stages. Each stage is skipped if it already finished. Capture and training also resume *within* the
stage: capture checkpoints after every batch, and training skips any SAE that's already saved.
The log prints images/min and an ETA during capture.
If the cell is interrupted, run it again.

In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', 'scripts/run_pipeline.py', '--root', ROOT,
       '--acts-dir', ACTS, '--n-images', str(N_IMAGES), '--site', SITE]
if RESUME_FROM:
    cmd += ['--resume-from', RESUME_FROM]
print(' '.join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
PIPELINE_OK = proc.wait() == 0
# Deliberately no exception: a background run that errors may not save its
# output, and that output is what lets the next run resume.
if PIPELINE_OK:
    print('\nPIPELINE COMPLETE')
else:
    print('\n' + '=' * 70 + '\nPIPELINE STOPPED -- a stage failed. Everything finished so far is\n'
          f'saved under {ROOT}. Logs: {ROOT}/logs. To continue, attach this version\n'
          'as input (Add Input -> Your Work) and Save & Run All again.\n' + '=' * 70)
    _st = f'{ROOT}/state/last_run.json'
    if os.path.exists(_st):
        import json as _json
        _f = (_json.load(open(_st)).get('failed') or {})
        if _f.get('log') and os.path.exists(_f['log']):
            print(f"\nlast 60 lines of {_f['log']}:\n")
            print(''.join(open(_f['log']).readlines()[-60:]))


## Results

In [ ]:
from IPython.display import Markdown, display
rep = f'{ROOT}/outputs/REPORT.md'
if os.path.exists(rep):
    display(Markdown(open(rep).read()))
else:
    print('no report yet -- the pipeline did not reach the report stage')


In [ ]:
import json, glob
from PIL import Image
ap = f'{ROOT}/outputs/atlas/atlas.json'
atlas = json.load(open(ap)) if os.path.exists(ap) else {'features': []}
print('Atlas -- correlational evidence only; labels are tested by the causal stages')
for f in atlas['features'][:8]:
    print(f"\nfeature {f['feature']}  fires on {f['fire_rate']:.0%} of prompts")
    print('   ', ' | '.join(p[:40] for p in f['top_prompts'][:3]))
    display(Image.open(f"{ROOT}/outputs/atlas/{f['image']}"))


In [ ]:
for p in sorted(glob.glob(f'{ROOT}/outputs/grids/*.png'))[:16]:
    print(p.split('/')[-1]); display(Image.open(p).resize((256, 256)))


## Package

In [ ]:
import shutil, subprocess
if PIPELINE_OK and not KEEP_ACTS and os.path.exists(ACTS):
    shutil.rmtree(ACTS); print('removed activations to keep the output small')
parts = [d for d in ('runs', 'outputs', 'logs') if os.path.exists(f'{ROOT}/{d}')]
if parts:
    subprocess.run(['tar', '-czf', f'{ROOT}/project_b_results.tgz', '-C', ROOT, *parts], check=True)
print('download:', f'{ROOT}/project_b_results.tgz')
